# Phase 2 POC — Predict optimal dwell map from coarse XRF (U-Net)

Goal: train a small U-Net that takes a coarse XRF composite (250 um) as input and predicts an "oracle" dwell map (derived from the fine 25 um signal). This is a supervised version of the temporal-adaptive strategy from script 07.

Pipeline:
1. Mount Google Drive (data is uploaded there)
2. Load (coarse, fine) HDF5 pairs for UA1_P1 / UB1_P1 / FP1_P1
3. Compute the oracle dwell map (proportional to fine signal)
4. Cut into 128×128 patches, augment (rotation + flip)
5. Define a small U-Net in PyTorch
6. Train, validate on a held-out sample
7. Compare predicted dwell map vs heuristic binary/linear from script 07

**Prerequisite**: upload the `data/` folder to Google Drive under `/MyDrive/SLAC-XRF-Project/data/`. The notebook will use Drive paths.

## 0. Setup

Runtime → Change runtime type → **GPU (T4)** if available (free Colab tier).

In [ ]:
!pip install -q h5py
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = '/content/drive/MyDrive/SLAC-XRF-Project'
DATA_DIR = os.path.join(PROJECT_ROOT, 'data', 'Data_May2026')
OUT_DIR = os.path.join(PROJECT_ROOT, 'phase2_ml', 'outputs')
os.makedirs(OUT_DIR, exist_ok=True)
print('Data dir:', DATA_DIR)
print('Existing files:', sorted(os.listdir(DATA_DIR))[:5], '...')

## 1. Load HDF5 helpers
Same logic as `scripts/utils/hdf5_reader.py`.

In [ ]:
import h5py
import numpy as np
from pathlib import Path

SKIP_CHANNELS = {'I0', 'I1', 'I0ZEBRA', 'I1ZEBRA', 'ICR', 'OCR',
                 'TIME', 'DTF', 'DTPCT', 'LASER'}

def load_xrf(filepath):
    with h5py.File(filepath, 'r') as f:
        mapdata = f['main/mapdata'][:]
        xdata   = f['main/xdata'][:]
        ydata   = f['main/ydata'][:]
        attrs   = dict(f['main'].attrs)
    labels = [s.decode() if isinstance(s, bytes) else s
              for s in attrs.get('labels', [])]
    return {'mapdata': mapdata, 'xdata': xdata, 'ydata': ydata,
            'labels': labels}

def composite(data, channels=None):
    elements = [l for l in data['labels'] if l not in SKIP_CHANNELS]
    if channels is not None:
        elements = [e for e in elements if e in channels]
    used = []
    for ch in elements:
        idx = data['labels'].index(ch)
        arr = data['mapdata'][:, :, idx]
        if arr.min() != arr.max():
            used.append(ch)
    indices = [data['labels'].index(ch) for ch in used]
    return data['mapdata'][:, :, indices].sum(axis=2), used

# Sample pair to test loading
coarse = load_xrf(os.path.join(DATA_DIR, 'SMW_UA1_P1_250um_10ms_12000_0_001.hdf5'))
fine   = load_xrf(os.path.join(DATA_DIR, 'SMW_UA1_P1_25um_10ms_12000_0_001.hdf5'))
coarse_comp, used = composite(coarse)
fine_comp,   _    = composite(fine, channels=used)
print('Coarse shape:', coarse_comp.shape, '  Fine shape:', fine_comp.shape)
print('Channels used:', used)

## 2. Oracle dwell map (target for the network)
Given the fine signal, what dwell map *would* maximize SNR on bright pixels at fixed total time?

For a Poisson detector, the SNR on a bright pixel after dwell `d` is `sqrt(signal * d)`. To equalize SNR across all bright pixels at fixed total time T, you allocate dwell **inversely proportional to signal** — but that's actually wrong for our goal.

Our goal is **maximize total photons collected on signal**: maximize `sum(signal * dwell)` under constraint `sum(dwell) = T`. With this objective, the optimal allocation is to put ALL time on the brightest pixel (degenerate). To regularize we cap each dwell at `dwell_high` and floor at `dwell_low`.

Practical compromise: **`dwell_oracle = dwell_low + (dwell_high - dwell_low) * normalized_signal`** — same shape as the linear strategy in script 07, but computed from the FINE signal (oracle, not coarse). The network's job is to predict this from the COARSE signal only.

In [ ]:
def oracle_dwell(fine_comp, dwell_low=1.0, dwell_high=50.0):
    lo, hi = np.quantile(fine_comp, [0.05, 0.95])
    normed = np.clip((fine_comp - lo) / max(hi - lo, 1.0), 0, 1)
    return dwell_low + normed * (dwell_high - dwell_low)

dwell_target = oracle_dwell(fine_comp)
print('Oracle dwell stats:')
print(f'  min={dwell_target.min():.2f}ms  max={dwell_target.max():.2f}ms  '
      f'median={np.median(dwell_target):.2f}ms')

# Visualize
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].imshow(coarse_comp, cmap='inferno', vmax=np.percentile(coarse_comp, 99))
ax[0].set_title('Coarse 250um (input)')
ax[1].imshow(fine_comp, cmap='inferno', vmax=np.percentile(fine_comp, 99.5))
ax[1].set_title('Fine 25um (used to build oracle)')
ax[2].imshow(dwell_target, cmap='viridis', vmin=1, vmax=50)
ax[2].set_title('Oracle dwell map (target)')
for a in ax: a.axis('off')
plt.tight_layout()
plt.show()

## 3. Resample coarse to fine grid, build (input, target) pairs

The U-Net needs input and target on the SAME grid. We upsample the coarse composite to the fine resolution (nearest neighbor — replicates each coarse pixel to its 10×10 = 100 sub-pixels for 250 → 25 µm).

In [ ]:
def upsample_to_fine(coarse_comp, coarse_xy, fine_xy):
    """Nearest-neighbor upsample of coarse_comp to the fine grid."""
    xc, yc = coarse_xy
    xf, yf = fine_xy
    def nearest(arr, vals):
        order = np.argsort(arr)
        pos = np.clip(np.searchsorted(arr[order], vals), 0, len(arr)-1)
        left = np.clip(pos-1, 0, len(arr)-1)
        choose_left = np.abs(arr[order][left] - vals) < np.abs(arr[order][pos] - vals)
        return order[np.where(choose_left, left, pos)]
    ix = nearest(xc, xf)
    iy = nearest(yc, yf)
    IY, IX = np.meshgrid(iy, ix, indexing='ij')
    return coarse_comp[IY, IX]

coarse_at_fine = upsample_to_fine(
    coarse_comp,
    (coarse['xdata'], coarse['ydata']),
    (fine['xdata'], fine['ydata']),
)
print('Coarse-upsampled shape:', coarse_at_fine.shape,
      ' must match fine shape:', fine_comp.shape)
assert coarse_at_fine.shape == fine_comp.shape

## 4. Cut into patches and build PyTorch Dataset

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

PATCH = 128
STRIDE = 64        # overlap → more patches

def make_patches(img_in, img_tgt, patch=PATCH, stride=STRIDE):
    H, W = img_in.shape
    patches = []
    for y in range(0, H - patch + 1, stride):
        for x in range(0, W - patch + 1, stride):
            patches.append((img_in[y:y+patch, x:x+patch],
                            img_tgt[y:y+patch, x:x+patch]))
    return patches

# normalize: coarse to [0, 1] via log1p + quantile
def normalize_input(x):
    x = np.log1p(x)
    return (x - x.min()) / max(x.max() - x.min(), 1e-6)

patches = make_patches(
    normalize_input(coarse_at_fine).astype(np.float32),
    dwell_target.astype(np.float32),   # target in ms, will be scaled in loss
)
print(f'Made {len(patches)} patches of {PATCH}×{PATCH}')

class PatchDS(Dataset):
    def __init__(self, patches, augment=False):
        self.patches = patches
        self.augment = augment
    def __len__(self):
        return len(self.patches)
    def __getitem__(self, i):
        a, b = self.patches[i]
        if self.augment:
            k = np.random.randint(4)
            a, b = np.rot90(a, k).copy(), np.rot90(b, k).copy()
            if np.random.rand() < 0.5:
                a, b = a[:, ::-1].copy(), b[:, ::-1].copy()
        return (torch.from_numpy(a)[None],   # add channel dim
                torch.from_numpy(b)[None])

# train/val split
split = int(0.85 * len(patches))
train_ds = PatchDS(patches[:split], augment=True)
val_ds   = PatchDS(patches[split:], augment=False)
print(f'Train: {len(train_ds)}  Val: {len(val_ds)}')

## 5. Small U-Net (PyTorch)
~1M parameters. Encoder/decoder with skip connections, single input channel (coarse composite), single output channel (predicted dwell in ms).

In [ ]:
import torch.nn as nn

def conv_block(c_in, c_out):
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, 3, padding=1), nn.ReLU(inplace=True),
        nn.Conv2d(c_out, c_out, 3, padding=1), nn.ReLU(inplace=True),
    )

class TinyUNet(nn.Module):
    def __init__(self, base=16):
        super().__init__()
        self.e1 = conv_block(1, base)
        self.e2 = conv_block(base, base*2)
        self.e3 = conv_block(base*2, base*4)
        self.b  = conv_block(base*4, base*8)
        self.u3 = nn.ConvTranspose2d(base*8, base*4, 2, stride=2)
        self.d3 = conv_block(base*8, base*4)
        self.u2 = nn.ConvTranspose2d(base*4, base*2, 2, stride=2)
        self.d2 = conv_block(base*4, base*2)
        self.u1 = nn.ConvTranspose2d(base*2, base, 2, stride=2)
        self.d1 = conv_block(base*2, base)
        self.out = nn.Conv2d(base, 1, 1)
        self.pool = nn.MaxPool2d(2)
    def forward(self, x):
        e1 = self.e1(x);            p1 = self.pool(e1)
        e2 = self.e2(p1);           p2 = self.pool(e2)
        e3 = self.e3(p2);           p3 = self.pool(e3)
        b  = self.b(p3)
        u3 = self.u3(b);            d3 = self.d3(torch.cat([u3, e3], 1))
        u2 = self.u2(d3);           d2 = self.d2(torch.cat([u2, e2], 1))
        u1 = self.u1(d2);           d1 = self.d1(torch.cat([u1, e1], 1))
        return self.out(d1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = TinyUNet().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'Model: {n_params:,} parameters on {device}')

## 6. Train (a few epochs to check the loop)

In [ ]:
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=8)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

for epoch in range(5):
    model.train()
    tr_loss = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        pred = model(x)
        loss = loss_fn(pred, y)
        opt.zero_grad(); loss.backward(); opt.step()
        tr_loss += loss.item()
    tr_loss /= len(train_loader)

    model.eval()
    va_loss = 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            va_loss += loss_fn(model(x), y).item()
    va_loss /= max(len(val_loader), 1)
    print(f'Epoch {epoch+1}: train MSE={tr_loss:.3f}  val MSE={va_loss:.3f}')

## 7. Visualize a predicted dwell map vs the oracle
Side by side on a validation patch.

In [ ]:
model.eval()
x, y = val_ds[0]
with torch.no_grad():
    pred = model(x[None].to(device))[0, 0].cpu().numpy()

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].imshow(x[0].numpy(), cmap='inferno')
ax[0].set_title('Input (normalized coarse)')
ax[1].imshow(y[0].numpy(), cmap='viridis', vmin=1, vmax=50)
ax[1].set_title('Oracle dwell (target)')
ax[2].imshow(pred, cmap='viridis', vmin=1, vmax=50)
ax[2].set_title('Predicted dwell')
for a in ax: a.axis('off')
plt.tight_layout()
plt.show()

## 8. Next steps

Once this POC trains correctly:
1. **Cross-sample training**: include UB1_P1 and FP1_P1 in the training set, hold one out for true test
2. **More augmentation**: gamma correction, intensity scaling
3. **Better loss**: weighted MSE that focuses on bright regions
4. **Compare against script 07**: predicted dwell vs binary heuristic, on real validation metrics (info ratio, bright-pixel boost)
5. **Save the model**: `torch.save(model.state_dict(), '/content/drive/MyDrive/SLAC-XRF-Project/phase2_ml/models/unet_v1.pt')`
6. **Integrate back** into `10_run_pipeline.py` as an alternative dwell allocator when a trained model is available